# 02 — Feature Engineering Demo
**Deep Learning for Cryptocurrency Price Forecasting**
*Muluh Penn Junior Patrick — M.Tech. Thesis 2026*

---
Demonstrates the full 149-feature engineering pipeline:
technical indicators, LTST decomposition, on-chain metrics,
sentiment, and macro features.


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True,
                     'grid.alpha': 0.3})
print('Environment ready.')


## 2.1 Load BTC daily data

In [ ]:
from src.data_collection.pipeline import load_raw_ohlcv

df = load_raw_ohlcv('BTC', '1d')
print(f'Raw BTC data: {df.shape}')
print(df[['open','high','low','close','volume']].tail(3))


## 2.2 Compute technical indicators (104 features)

In [ ]:
from src.preprocessing.technical_indicators import TechnicalIndicatorComputer

tic = TechnicalIndicatorComputer()
df_ta = tic.compute(df)
ta_cols = [c for c in df_ta.columns if c not in df.columns]
print(f'Added {len(ta_cols)} technical indicators. Total features: {df_ta.shape[1]}')
print('Sample features:', ta_cols[:10])


## 2.3 LTST decomposition (35 features)

In [ ]:
from src.preprocessing.ltst_decomposition import LTSTDecomposer

decomp = LTSTDecomposer(methods=['ma'])
df_ltst = decomp.decompose(df_ta)
ltst_cols = [c for c in df_ltst.columns if c not in df_ta.columns]
print(f'Added {len(ltst_cols)} LTST decomposition features.')

# Visualise trend vs price
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(df_ltst.index, df_ltst['close'], color='#1a1a2e', linewidth=1, label='BTC Close')
if 'ltt_200' in df_ltst.columns:
    axes[0].plot(df_ltst.index, df_ltst['ltt_200'], color='#D85A30',
                 linewidth=1.5, label='LTT-200', alpha=0.8)
axes[0].set_ylabel('Price (USD)'); axes[0].legend(fontsize=8)
axes[0].set_title('LTST Decomposition — BTC 1d')

if 'stt_20' in df_ltst.columns:
    axes[1].plot(df_ltst.index, df_ltst['stt_20'], color='#1D9E75', linewidth=1, label='STT-20')
    axes[1].set_ylabel('Short-term trend'); axes[1].legend(fontsize=8)

if 'ltt_200' in df_ltst.columns and 'close' in df_ltst.columns:
    residual = df_ltst['close'] - df_ltst['ltt_200']
    axes[2].fill_between(df_ltst.index, residual, 0,
                         where=residual>=0, color='#1D9E75', alpha=0.5, label='Above trend')
    axes[2].fill_between(df_ltst.index, residual, 0,
                         where=residual<0, color='#D85A30', alpha=0.5, label='Below trend')
    axes[2].set_ylabel('Residual (Price − LTT-200)'); axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()


## 2.4 Feature category breakdown

In [ ]:
# Count features by category prefix
categories = {
    'OHLCV base':        ['open','high','low','close','volume'],
    'Trend':             [c for c in df_ltst.columns if any(c.startswith(p)
                          for p in ['ema','sma','dema','tema','trix','cci','aroon','adx'])],
    'Momentum':          [c for c in df_ltst.columns if any(c.startswith(p)
                          for p in ['rsi','stoch','macd','roc','williams','ultimate'])],
    'Volatility':        [c for c in df_ltst.columns if any(c.startswith(p)
                          for p in ['atr','bb_','kc_','donchian','true_range'])],
    'Volume':            [c for c in df_ltst.columns if any(c.startswith(p)
                          for p in ['obv','vwap','mfi','cmf','fi_','ease','vpt'])],
    'Derived/cross':     [c for c in df_ltst.columns if any(c.startswith(p)
                          for p in ['price_vs','log_return','hlcc4','ohlc4'])],
    'LTST decomposition':[c for c in df_ltst.columns if any(c.startswith(p)
                          for p in ['ltt_','stt_','ma_res','hp_','stl_'])],
}
print(f'{"Category":<24} {"Features":>8}')
print('─' * 34)
total = 0
for cat, cols in categories.items():
    n = len(cols)
    total += n
    print(f'{cat:<24} {n:>8}')
print('─' * 34)
print(f'{"TOTAL":<24} {total:>8}')
print(f'\nActual total columns in DataFrame: {df_ltst.shape[1]}')


## 2.5 Zero-leakage normalization

In [ ]:
from src.preprocessing.normalizer import CryptoNormalizer
from src.training.walk_forward_cv import temporal_split

train_df, val_df, test_df = temporal_split(df_ltst)
print(f'Train: {len(train_df)} rows | Val: {len(val_df)} | Test: {len(test_df)}')

normalizer = CryptoNormalizer(method='minmax')
normalizer.fit(train_df)   # fit on train ONLY — no leakage
train_norm = normalizer.transform(train_df)
val_norm   = normalizer.transform(val_df)
test_norm  = normalizer.transform(test_df)
print(f'Normalizer fitted on {len(train_df)} training rows. '
      f'Applied to val and test without re-fitting.')


## 2.6 Sequence construction (sliding window)

In [ ]:
from src.preprocessing.sequence_builder import SequenceBuilder

builder = SequenceBuilder(seq_len=90, horizon=1, stride=1)
X_train, y_train = builder.build(train_norm, target_col='log_returns')
X_val,   y_val   = builder.build(val_norm,   target_col='log_returns')
X_test,  y_test  = builder.build(test_norm,  target_col='log_returns')
print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}    y_val:   {y_val.shape}')
print(f'X_test:  {X_test.shape}   y_test:  {y_test.shape}')
print(f'\nInput tensor shape per sample: (seq_len={X_train.shape[1]}, '
      f'features={X_train.shape[2]})')


## 2.7 Summary
The feature engineering pipeline produces **149 features** per timestep:
- 104 technical indicators (trend, momentum, volatility, volume, derived)
- 35 LTST decomposition features (MA-based trend/residual decomposition)
- 10 additional market/on-chain/sentiment/macro features

Zero-leakage protocol: normalizer fitted on training set only.
Sequences built with 90-day lookback window and 1-day forecast horizon.

**→ Next: Baseline Models** (`03_baseline_models.ipynb`)
